# 3. Salary Prediction Model

This notebook predicts annual technology salary in USD using job and company characteristics.

The model is educational. The data is global, geographically imbalanced, and does not include job-description skills.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

DATA_PATH = Path("../data/ds_salaries_clean.csv")
df = pd.read_csv(DATA_PATH)

print(f"Loaded {len(df)} rows and {df.shape[1]} columns")
display(df.head())

## Prepare the prediction data

The target is `salary_in_usd`. Salary-derived columns are excluded to prevent target leakage.

In [ ]:
target = "salary_in_usd"
feature_columns = [
    "work_year",
    "experience_level",
    "employment_type",
    "job_title",
    "remote_ratio",
    "company_size",
    "employee_continent",
    "company_continent",
 ]

categorical_features = [
    "experience_level",
    "employment_type",
    "job_title",
    "company_size",
    "employee_continent",
    "company_continent",
 ]
numeric_features = ["work_year", "remote_ratio"]

model_df = df[feature_columns + [target]].dropna().copy()
X = model_df[feature_columns]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
 )

print(f"Model rows: {len(model_df)}")
print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

## Compare with a baseline

The baseline predicts the median salary from the training data. The random-forest model must improve on this simple reference.

In [ ]:
def regression_metrics(actual, predicted):
    return {
        "MAE_USD": mean_absolute_error(actual, predicted),
        "RMSE_USD": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
    }

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
baseline_metrics = regression_metrics(y_test, baseline_predictions)

display(pd.Series(baseline_metrics, name="Baseline"))

## Train and evaluate the model

Categorical columns are one-hot encoded inside a pipeline. This prevents preprocessing from learning from the test set.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ]
 )

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
 )

model.fit(X_train, y_train)
model_predictions = model.predict(X_test)
model_metrics = regression_metrics(y_test, model_predictions)

metrics = pd.DataFrame(
    [baseline_metrics, model_metrics],
    index=["Baseline", "Random forest"],
 )
display(metrics)

## Inspect errors and feature importance

In [ ]:
results = X_test.copy()
results["actual_salary_usd"] = y_test
results["predicted_salary_usd"] = model_predictions
results["absolute_error_usd"] = (
    results["actual_salary_usd"] - results["predicted_salary_usd"]
 ).abs()

display(
    results.sort_values("absolute_error_usd", ascending=False)[
        ["actual_salary_usd", "predicted_salary_usd", "absolute_error_usd"]
    ].head(10)
 )

fitted_preprocessor = model.named_steps["preprocessor"]
fitted_regressor = model.named_steps["regressor"]
encoded_features = fitted_preprocessor.get_feature_names_out()

importance = pd.Series(
    fitted_regressor.feature_importances_,
    index=encoded_features,
    name="importance",
 ).sort_values(ascending=False).head(15).sort_values()

display(importance.to_frame())
importance.plot(kind="barh", figsize=(10, 6), legend=False)
plt.title("Top Salary Model Features")
plt.xlabel("Random-forest importance")
plt.tight_layout()
plt.show()

## Example prediction and limitations

The prediction below demonstrates model usage. It is not a market guarantee, especially because this dataset is global and does not contain skills or job descriptions.

In [ ]:
example_job = pd.DataFrame([{
    "work_year": 2022,
    "experience_level": "SE",
    "employment_type": "FT",
    "job_title": "Data Scientist",
    "remote_ratio": 100,
    "company_size": "M",
    "employee_continent": "North America",
    "company_continent": "North America",
}])

example_prediction = model.predict(example_job)[0]
print(f"Estimated salary: ${example_prediction:,.0f} per year")

- The data is heavily concentrated in North America and Europe.
- Salary depends on skills, industry, company, benefits, and negotiation, which are not included.
- A random split can place similar roles in both training and test sets.
- Cross-validation and a larger Tunisian job-posting dataset would improve the analysis.
- Job descriptions would enable skill extraction and skill-based salary prediction.